In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import warnings

warnings.filterwarnings('ignore')

In [ ]:
df_app = pd.read_csv('RAW_LAG_APP.csv')

In [ ]:
df_app

In [ ]:
df_app.columns

In [ ]:
app_features = ['phq1', 'phq2', 'meal_morning', 'meal_lunch',
       'meal_dinner', 'meal_night', 'sleep_satisfy', 'sleep_disturb',
       'alcohol_when_morning', 'alcohol_when_lunch', 'alcohol_when_dinner',
       'alcohol_when_night', 'amount_beer_glass', 'amount_soju_glass',
       'amount_wine_glass', 'amount_whiskey_glass', 'amount_rice_glass',
       'alcohol_amount_etc_True', 'alcohol_why_stress_True',
       'alcohol_why_social_True', 'alcohol_why_happy_True',
       'alcohol_why_sleep_True', 'alcohol_why_lonely_True',
       'alcohol_why_unavoidable_True', 'alcohol_why_habit_True',
       'alcohol_why_pairing_True', 'alcohol_with_whom_other',
       'alcohol_with_whom_with_colleagues', 'alcohol_with_whom_with_family',
       'alcohol_with_whom_with_friends', 'alcohol_with_whom_with_partner',
       'alcohol_how_long_less_than_2_hours',
       'alcohol_how_long_less_than_4_hours',
       'alcohol_how_long_more_than_4_hours', 'alcohol_how_long_other',
       'alcohol_where_etc', 'alcohol_where_friends_home', 'alcohol_where_home',
       'alcohol_where_outdoor', 'alcohol_where_pub',
       'alcohol_where_restaurant', 'alcohol_blackout_never',
       'alcohol_blackout_other', 'alcohol_blackout_partially',
       'sleep_minute_adjusted']

In [ ]:
len(app_features)

In [ ]:
df_app = df_app.sort_values(by = ['user_id', 'date'])


In [ ]:
df_app_reduced = df_app[['user_order', 'day_order', 'tmrw_alcohol_desire']]

In [ ]:
df_app_reduced

In [ ]:
def compute_train_mean(user_df):
    user_df = user_df.sort_values('day_order')
    
    return user_df.iloc[:-8].mean()

In [ ]:
df_mean = df_app_reduced.groupby('user_order').apply(compute_train_mean)

In [ ]:
df_mean

In [ ]:
df_baseline = pd.read_csv('RAW_Scaled_Baseline.csv')

In [ ]:
df_baseline


In [ ]:
df_baseline.columns

In [ ]:
df_baseline['tmrw_alcohol_desire'] = df_mean.tmrw_alcohol_desire.values

In [ ]:
df_baseline

## Feature Selection

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

In [ ]:
y = df_baseline.tmrw_alcohol_desire

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(df_baseline.drop(columns = ['대상자번호', '하루짠ID', 'tmrw_alcohol_desire'], axis = 1, errors = 'ignore'), y)


In [ ]:
selector = SelectFromModel(rf, max_features=42, prefit=True)

In [ ]:
selected_baseline_features = df_baseline.drop(columns = ['대상자번호', '하루짠ID', 'tmrw_alcohol_desire'], axis = 1, errors = 'ignore').columns[selector.get_support()]

In [ ]:
selected_baseline_features

In [ ]:
df_baseline_feature_selection = df_baseline[selected_baseline_features]

In [ ]:
df_baseline_feature_selection['tmrw_alcohol_desire'] = df_baseline.tmrw_alcohol_desire

In [ ]:
df_baseline_feature_selection[['대상자번호', '하루짠ID']] = df_baseline[['대상자번호', '하루짠ID']]

In [ ]:
cols = list(df_baseline_feature_selection.columns)

cols.remove('대상자번호')
cols.remove('하루짠ID')

cols.insert(0, '대상자번호')
cols.insert(1, '하루짠ID')

df_baseline_feature_selection = df_baseline_feature_selection[cols]

In [ ]:
df_baseline_feature_selection

In [ ]:
df_baseline_feature_selection.to_csv('Baseline_FeatureSelection copy copy.csv', index=False)